# Deforestation and Recovery Balance — Managed Forest of Quebec

This notebook extracts annual Sentinel-2 composites over the Abitibi-Temiscamingue administrative region (managed public forest, Quebec), detects year-over-year change (dNBR), and computes the net balance between lost and recovering forest cover per spatial unit. It also derives aboveground carbon flux from ESA CCI Biomass and crosses it with the change classification.

**Pipeline:** GEE extraction -> annual NDVI/NBR composites -> change detection -> spatial aggregation -> net balance -> carbon flux (ESA CCI Biomass) -> carbon source/sink in change zones.

In [ ]:
import sys
sys.path.append('../src')

import ee
import geemap
import pandas as pd

from gee_utils import init_ee, build_annual_stack
from change_detection import build_change_series, aggregate_by_units

PROJECT = "your-gee-project-id"
init_ee(PROJECT)

## 1. Study area: Abitibi-Temiscamingue administrative region

Source: "Decoupages administratifs" layer from Donnees Quebec / MRNF (province-wide administrative divisions, 1:1,000,000 scale), filtered to the Abitibi-Temiscamingue region. Download it locally first (see README) and place it under `data/`.

In [ ]:
import geopandas as gpd
import geemap
import glob

# The region polygon layer is regio_s.shp ('_s' = surface/polygons,
# '_l' = boundary lines only) inside the extracted BDGA_1M archive.
shp_candidates = glob.glob("../data/decoupages/**/regio_s.shp", recursive=True)
print("Found:", shp_candidates)
SHP_PATH = shp_candidates[0]

In [ ]:
gdf = gpd.read_file(SHP_PATH)
print("Columns:", list(gdf.columns))
gdf.head()

In [ ]:
# Identify the field holding the region name
# (check unique values to confirm the exact column and value)
REGION_NAME_FIELD = "NOM"  # adjust based on the columns printed above

print(gdf[REGION_NAME_FIELD].unique())

In [ ]:
# Filter the region and convert it to a GEE AOI
gdf_region = gdf[gdf[REGION_NAME_FIELD].str.contains("Abitibi", case=False, na=False)]
gdf_region = gdf_region.to_crs("EPSG:4326")

aoi_fc = geemap.geopandas_to_ee(gdf_region)
AOI = aoi_fc.geometry()

Map = geemap.Map()
Map.centerObject(AOI, 8)
Map.addLayer(AOI, {}, "AOI - Abitibi-Temiscamingue")
Map

## 2. Annual composites (NDVI/NBR)

In [ ]:
YEARS = range(2017, 2026)
annual_stack = build_annual_stack(YEARS, AOI)
print(f"Composites generated: {annual_stack.size().getInfo()}")

In [ ]:
# Quick preview of the latest composite (NBR)
last_img = ee.Image(annual_stack.sort('year', False).first())
nbr_vis = {"min": -0.5, "max": 0.8, "palette": ["red", "white", "green"]}

Map2 = geemap.Map()
Map2.centerObject(AOI, 9)
Map2.addLayer(last_img.select('NBR'), nbr_vis, "NBR - latest year")
Map2

## 3. Year-over-year change detection

In [ ]:
change_pairs = build_change_series(annual_stack, band="NBR")
print([year for year, _ in change_pairs])

## 4. Spatial aggregation and net balance

Requires a spatial unit layer (`units_fc`) with a `unit_id` property — e.g. a hexagon grid or an MRC subdivision. Placeholder: generate a grid with `geemap` or upload your own asset.

In [ ]:
# Placeholder: replace with the real spatial unit layer
# units_fc = ee.FeatureCollection("projects/your-gee-project-id/assets/hex_grid")

results = []
for year, classified in change_pairs:
    # stats = aggregate_by_units(classified, units_fc, scale=10)
    # df_year = geemap.ee_to_df(stats)
    # df_year['year'] = year
    # results.append(df_year)
    pass

# balance_df = pd.concat(results, ignore_index=True)
# balance_df.to_csv('../figures/net_balance_by_unit.csv', index=False)

## 5. Result: cumulative net balance

Final chart for the LinkedIn post: annual loss vs recovery area, and cumulative net balance.

In [ ]:
import matplotlib.pyplot as plt

# balance_summary = balance_df.groupby('year')[['loss_ha', 'recovery_ha', 'net_balance_ha']].sum()
# balance_summary['net_balance_cumsum'] = balance_summary['net_balance_ha'].cumsum()

# fig, ax = plt.subplots(figsize=(9, 5))
# balance_summary[['loss_ha', 'recovery_ha']].plot(kind='bar', ax=ax)
# ax.set_ylabel('Hectares')
# ax.set_title('Annual loss vs recovery — managed forest of Quebec')
# plt.tight_layout()
# plt.savefig('../figures/annual_loss_recovery.png', dpi=200)
# plt.show()

## 6. Aboveground carbon flux (ESA CCI Biomass)

ESA CCI Biomass v6.0 provides continuous, gap-free annual aboveground biomass (AGB) maps (2007, 2010, 2015-2022) — unlike GEDI's sparse orbital-track sampling. Comparing two years directly yields a carbon flux map: positive values are carbon sinks (net gain), negative values are carbon sources (net loss). Citation: Santoro & Cartus (2025), ESA CCI Biomass v6.0.

In [ ]:
from biomass_utils import (
    list_collection_dates,
    get_carbon_density,
    carbon_flux,
    classify_flux,
    aggregate_flux_by_units,
)

# Diagnostic: confirm how years are indexed in the collection before trusting
# the calendarRange filter in biomass_utils.get_agb
print(list_collection_dates().getInfo())

In [ ]:
# Sentinel-2 covers 2017-2025; ESA CCI Biomass tops out at 2022 -> use the
# widest available overlap for the flux comparison.
YEAR_T0 = 2017
YEAR_T1 = 2022

flux = carbon_flux(YEAR_T0, YEAR_T1, AOI)

flux_vis = {"min": -20, "max": 20, "palette": ["red", "white", "darkgreen"]}
Map3 = geemap.Map()
Map3.centerObject(AOI, 8)
Map3.addLayer(flux, flux_vis, f"Carbon flux {YEAR_T0}-{YEAR_T1} (Mg C/ha)")
Map3

## 7. Carbon flux classification and aggregation by spatial unit

In [ ]:
classified_flux = classify_flux(flux, source_threshold=-5, sink_threshold=5)

# flux_by_unit = aggregate_flux_by_units(flux, units_fc, scale=100)
# flux_df = geemap.ee_to_df(flux_by_unit)
# flux_df.to_csv('../figures/carbon_flux_by_unit.csv', index=False)

# Total estimated carbon stock change (Mg) across the whole AOI
# pixel_area_ha = ee.Image.pixelArea().divide(10000)
# total_change = flux.multiply(pixel_area_ha).reduceRegion(
#     reducer=ee.Reducer.sum(), geometry=AOI, scale=100, maxPixels=1e13
# )
# print('Estimated total carbon stock change (Mg):', total_change.getInfo())

## 8. Combined result for LinkedIn

Two final visuals: (1) net cover loss/recovery balance map (Sentinel-2), (2) carbon flux map (ESA CCI Biomass) — message: which areas are net carbon sources vs sinks, and how that lines up with the cover-loss zones.